# 01 — Baseline: does the pipeline work at all?

**Talk role:** This is the opener. "50 lines and you've got multimodal video search." Walk through this notebook on stage, get the first few results, audience is hooked. Then the rest of the talk is everything you discovered after this notebook worked.

Run `python scripts/ingest.py --clips ./clips --strategy scene` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))

from embed_jina import JinaClient, EmbedConfig
from index_elastic import es_client, knn_search, index_name

jina = JinaClient()
es = es_client()
INDEX = index_name('scene', 1024, 'float')
print(f'Index: {INDEX}  →  {es.count(index=INDEX)["count"]} chunks')

In [ ]:
def search(query: str, k: int = 5):
    qvec = jina.embed([query], task='retrieval.query', config=EmbedConfig())[0]
    hits = knn_search(es, INDEX, qvec, k=k)
    for i, h in enumerate(hits):
        print(f'  #{i+1}  {h["_score"]:.3f}  {h["chunk_id"]:40s}  {h["duration"]:5.1f}s')
    return hits

In [ ]:
search('aerial shot of a forest')

In [ ]:
search('warm sunset light')

In [ ]:
search('person typing on a laptop')